# WRDS Extract 01: S&P 500 Membership

Run the connection cell first. That should trigger the normal WRDS / Duo authentication flow in the same way as a simple `wrds.Connection()` call.

In [2]:
from pathlib import Path

import pandas as pd
import wrds
from pandas.tseries.offsets import MonthEnd

In [3]:
start_date = pd.Timestamp("2000-01-01")
end_date = pd.Timestamp("2025-12-31")
outdir = Path("data/raw")
membership_table = "crsp.dsp500list"

# Leave these as None if you want WRDS to use its default login flow.
wrds_username = None
wrds_password = None

outdir.mkdir(parents=True, exist_ok=True)
start_date, end_date, outdir

(Timestamp('2000-01-01 00:00:00'),
 Timestamp('2025-12-31 00:00:00'),
 PosixPath('data/raw'))

In [4]:
if wrds_username is not None and wrds_password is not None:
    db = wrds.Connection(wrds_username=wrds_username, wrds_password=wrds_password)
elif wrds_username is not None:
    db = wrds.Connection(wrds_username=wrds_username)
else:
    db = wrds.Connection()

db

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [5]:
membership_sql = f"""
    SELECT DISTINCT
        permno::integer AS permno,
        GREATEST(\"start\"::date, CAST(%(start_date)s AS date)) AS start_date,
        LEAST(
            COALESCE(ending::date, CAST(%(end_date)s AS date)),
            CAST(%(end_date)s AS date)
        ) AS ending_date
    FROM {membership_table}
    WHERE permno IS NOT NULL
      AND \"start\"::date <= CAST(%(end_date)s AS date)
      AND COALESCE(ending::date, CAST(%(end_date)s AS date)) >= CAST(%(start_date)s AS date)
    ORDER BY permno, start_date, ending_date
"""

membership = db.raw_sql(
    membership_sql,
    params={
        "start_date": start_date.date(),
        "end_date": end_date.date(),
    },
)

if membership.empty:
    raise RuntimeError(
        f"Membership query returned zero rows from {membership_table!r}. "
        "The table may be unavailable for this WRDS account."
    )

membership.head()

,permno,start_date,ending_date
0,10078,2000-01-01,2010-01-28
1,10104,2000-01-01,2024-12-31
2,10107,2000-01-01,2024-12-31
3,10108,2002-07-22,2005-08-11
4,10137,2000-12-11,2011-02-25


In [ ]:
membership["permno"] = pd.to_numeric(membership["permno"], errors="coerce").astype(
    "Int64"
)
membership["start_date"] = pd.to_datetime(membership["start_date"], errors="coerce")
membership["ending_date"] = pd.to_datetime(membership["ending_date"], errors="coerce")
membership = membership.dropna(subset=["permno", "start_date", "ending_date"]).copy()
membership["permno"] = membership["permno"].astype(int)

if membership.empty:
    raise RuntimeError(
        "Membership rows were returned, but none had valid permno/start/end values."
    )

membership.shape

(1115, 3)

In [ ]:
def iter_month_ends(start_dt: pd.Timestamp, end_dt: pd.Timestamp) -> pd.DatetimeIndex:
    first_month_end = start_dt + MonthEnd(0)
    if first_month_end > end_dt:
        return pd.DatetimeIndex([])
    return pd.date_range(first_month_end, end_dt, freq=MonthEnd())


pieces = []
for row in membership.itertuples(index=False):
    month_ends = iter_month_ends(row.start_date, row.ending_date)
    if len(month_ends) == 0:
        continue
    pieces.append(pd.DataFrame({"permno": int(row.permno), "month_end": month_ends}))

if not pieces:
    raise RuntimeError(
        "No valid month-end membership observations could be constructed."
    )

membership_monthly = (
    pd.concat(pieces, ignore_index=True)
    .drop_duplicates(subset=["permno", "month_end"])
    .sort_values(["month_end", "permno"])
    .reset_index(drop=True)
)

membership_monthly.head()

,permno,month_end
0,10078,2000-01-31
1,10104,2000-01-31
2,10107,2000-01-31
3,10138,2000-01-31
4,10145,2000-01-31


In [8]:
membership_outfile = outdir / "sp500_membership_monthly.parquet"
membership_monthly.to_parquet(membership_outfile, engine="pyarrow", index=False)

print(f"saved: {membership_outfile}")
print(f"rows: {len(membership_monthly):,}")
print(f"permnos: {membership_monthly['permno'].nunique():,}")

saved: data/raw/sp500_membership_monthly.parquet
rows: 150,492
permnos: 1,062


In [ ]:
db.close()

: 